# Chapter 2 — First-Order Logic and Reasoning
### Notebook 4 · Agentic lab — formalisation, graded semantically

*Book reference: Extends Ch. 2*

An agent that turns English into logic. What makes this lab different from most LLM evaluation: the grader is a **decision procedure**, not a string comparison and not a judge.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch02_toolkit as fol
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch02_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Grade a generative task **semantically**, using Chapter 2's own entailment checker instead of string overlap.
2. Turn a countermodel into optimiser feedback.
3. Model **proof search** as an MDP and solve it exactly.
4. Watch GEPA learn the quantifier rules in two distinct stages.

> **Prerequisite:** Chapter 1's agentic lab, which introduced tools, metrics, MDPs, GEPA and skills. Here the task changes and the discipline does not.

## 1. Why this task is unusually well-posed

Most generative tasks are graded by string overlap (crude) or by a judge (expensive, and itself unvalidated). Formalisation has neither problem: two formulas are equivalent when they have the same models, and Notebook 1 built the machinery to check that.

So `forall x (P(x) -> Q(x))` and `~exists x (P(x) & ~Q(x))` both earn full marks, while a formula that merely *looks* similar earns none.

In [4]:
a = 'forall x (Human(x) -> Mortal(x))'
b = '~exists x (Human(x) & ~Mortal(x))'      # different string, same meaning
c = 'forall x (Human(x) & Mortal(x))'        # similar string, different meaning
print(f'{a}\n  == {b} ?', A.equivalent(fol.parse(a), fol.parse(b)))
print(f'{a}\n  == {c} ?', A.equivalent(fol.parse(a), fol.parse(c)))

forall x (Human(x) -> Mortal(x))
  == ~exists x (Human(x) & ~Mortal(x)) ? True
forall x (Human(x) -> Mortal(x))
  == forall x (Human(x) & Mortal(x)) ? False


## 2. Tools

The agent gets the reasoning services as function tools. Note `check_entailment` returns a **countermodel** on failure — a tool that explains its 'no' is worth far more to an agent than one that merely reports it.

In [5]:
ctx = A.Ch2Context()
tools = {t.name: t for t in A.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:20s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":20s} {t.description.splitlines()[0]}')

parse_formula        ['text']
                     Check that a formula parses, and report its predicates and free variables.
assert_premise       ['text']
                     Add a formula to the working premise set.
check_entailment     ['conclusion', 'max_size']
                     Do the asserted premises entail this conclusion over small finite models?
find_countermodel    ['premise', 'conclusion', 'max_size']
                     Search for a model making `premise` true and `conclusion` false.
prove                ['conclusion', 'constants']
                     Attempt a ground resolution refutation of premises + negated conclusion.
list_pitfalls        []
                     List the classic formalisation mistakes and how to avoid each one.


In [6]:
print(tools['assert_premise'].invoke({'text': 'forall x (Human(x) -> Mortal(x))'}))
print(tools['assert_premise'].invoke({'text': 'Human(Socrates)'}))
print(tools['check_entailment'].invoke({'conclusion': 'Mortal(Socrates)'}))
print(tools['check_entailment'].invoke({'conclusion': 'Mortal(Plato)'}))
print()
print(tools['prove'].invoke({'conclusion': 'Mortal(Socrates)'}))
print('\ntool calls logged:', ctx.log.names())

{"premises": ["forall x (Human(x) -> Mortal(x))"]}
{"premises": ["forall x (Human(x) -> Mortal(x))", "Human(Socrates)"]}
{"entails": true, "countermodel": null, "searched_domains_up_to": 3}
{"entails": false, "countermodel": "domain = {e0, e1}\nPlato = e1\nSocrates = e0\nHuman = {e0}\nMortal = {e0}", "searched_domains_up_to": 3}

{"proved": true, "resolution_steps": 4, "domain": ["Socrates"], "trace": [[["Human(Socrates)"], ["Mortal(Socrates)", "~Human(Socrates)"], ["Mortal(Socrates)"]], [["~Mortal(Socrates)"], ["Mortal(Socrates)", "~Human(Socrates)"], ["~Human(Socrates)"]], [["Human(Socrates)"], ["~Human(Socrates)"], []]]}

tool calls logged: ['assert_premise', 'assert_premise', 'check_entailment', 'check_entailment', 'prove']


## 3. The dataset and the semantic metric

Ten statements, split by item. The metric is staged, and the staging is deliberate:

| outcome | score |
|---|---|
| does not parse | 0.00 |
| parses, wrong meaning | 0.25 |
| semantically equivalent | 1.00 |

A flat zero for both failure modes would tell the optimiser only *that* it failed. Partial credit for well-formedness gives it a gradient to climb in two steps — first *be parseable*, then *be right*.

In [7]:
train, dev = A.build_dataset('train'), A.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')
for e in dev:
    print(f'  {e.id:28s} {e.statement:45s} {e.gold_formula}')

train 6, dev 4
  every-branch-part-tree       Every branch is part of some tree.            forall x (Branch(x) -> exists y (Tree(y) & PartOf(x, y)))
  some-carnivore-eats-impala   Some carnivore eats an impala.                exists x (Carnivore(x) & exists y (Impala(y) & Eats(x, y)))
  all-trees-plants             All trees are plants.                         forall x (Tree(x) -> Plant(x))
  someone-teaches-everything   There is someone who teaches everything.      exists x forall y Teaches(x, y)


In [8]:
lm = llm.configure_dspy(A.FOL_RULEBOOK, A.fol_responder)
baseline = A.FormalisationProgram()
example = train[0]
pred = baseline(**example.inputs())
report = A.translation_scorer(example, pred)
print('statement :', example.statement)
print('produced  :', repr(pred.formula))
print('score     :', report.score)
for n in report.notes:
    print('  ', n)

statement : Every human is mortal.
produced  : 'The formula is: forall x (Human(x) & Mortal(x)).'
score     : 0.0
   Formula does not parse (cannot tokenise at position 14: ': forall x ('). Answer with the formula alone, in ASCII FOL such as 'forall x (P(x) -> Q(x))' -- no surrounding prose.


### Feedback carries the countermodel

Once the formula parses, a wrong reading is diagnosed by *name* and witnessed by a model. This is what the GEPA reflection step reads:

In [9]:
mid = A.FormalisationProgram(
    A.BASELINE_INSTRUCTION + '\n- RULE emit-parseable-formula: answer with the formula alone')
gepa_metric = ev.make_gepa_metric(A.translation_scorer, A.FOL_RULEBOOK)
fb = gepa_metric(example, mid(**example.inputs()))
print('score:', fb.score)
print(fb.feedback)

score: 0.25
Well-formed, but not equivalent to the intended reading.
  produced: forall x (Human(x) & Mortal(x))
  intended: forall x (Human(x) -> Mortal(x))
  this is the 'universal-uses-implication' error: The conjunctive reading says *everything in the domain* is a student. Universal restriction is always material implication.
  witnessed by this model, where the two readings differ:
    domain = {e0}
    Human = {}
    Mortal = {}
MISSING RULE universal-uses-implication: Translate 'every/all X is Y' as forall x (X(x) -> Y(x)); a conjunction under a universal claims every object in the domain is an X.
Score: 0.250


## 4. Optimising with GEPA — in two stages

The baseline scores **0.0**: it wraps every formula in prose, so nothing parses. Watch the optimiser first learn to emit a bare formula, then learn the quantifier semantics.

In [10]:
before = ev.evaluate_dataset(baseline, dev, A.translation_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

BEFORE: 0.0 {'emit-parseable-formula': 4}


In [11]:
reflect = llm.reflection_lm(A.FOL_RULEBOOK, A.fol_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(A.FormalisationProgram(), tuned, dev, A.translation_scorer)
print(result.report())

2026/08/15 19:51:17 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 80 metric calls of the program. This amounts to 6.67 full evals on the train+val set.


2026/08/15 19:51:17 INFO dspy.teleprompt.gepa.gepa: Using 6 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/80 [00:00<?, ?rollouts/s]

2026/08/15 19:51:17 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 6 (0.0%)


2026/08/15 19:51:17 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0


...

2026/08/15 19:51:18 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/15 19:51:18 INFO dspy.teleprompt.gepa.gepa: Iteration 21: All subsample scores perfect. Skipping.


2026/08/15 19:51:18 INFO dspy.teleprompt.gepa.gepa: Iteration 21: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 78/80 [00:00<00:00, 155.88rollouts/s]

mean score  0.000  ->  1.000   (delta +1.000)
violations  {'emit-parseable-formula': 4}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,6 @@
 Translate the statement into first-order logic.
+- RULE emit-parseable-formula: Answer with a formula in the ASCII syntax: forall/exists, ~ & | -> <->, predicates as Name(arg). Nothing else.
+- RULE negation-scope: Translate 'no X is Y' as forall x (X(x) -> ~Y(x)), putting the negation inside the scope of the universal.
+- RULE universal-uses-implication: Translate 'every/all X is Y' as forall x (X(x) -> Y(x)); a conjunction under a universal claims every object in the domain is an X.
+- RULE existential-uses-conjunction: Translate 'some X is Y' as exists x (X(x) & Y(x)); an implication under an existential is satisfied by any non-X and asserts almost nothing.
+- RULE quantifier-order-matters: Keep the quantifier order of the English: 'everyone R something' is forall x exists y, never exists y forall x.

In [12]:
found = A.FOL_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(A.FOL_RULEBOOK.ids) - found))

rules discovered: ['emit-parseable-formula', 'existential-uses-conjunction', 'negation-scope', 'quantifier-order-matters', 'universal-uses-implication']
rules missed    : []


## 5. Proof search as an MDP

Chapter 1's MDP bought information; Chapter 4's built an artefact. This one **searches**.

| | Chapter 2 proof search |
|---|---|
| **S** | the set of clauses derived so far, plus whether a verdict was given |
| **A** | derive a resolvent, or claim entailed / not-entailed |
| **T** | deterministic — resolution is a function of the clauses |
| **R** | −cost per resolution; **+1 for a justified correct verdict**, −1 otherwise |

The word *justified* is doing real work: claiming entailment is only rewarded once the empty clause is actually derived. Guessing right is not the same as proving.

In [13]:
prem = [fol.parse('forall x (Human(x) -> Mortal(x))'), fol.parse('Human(Socrates)')]
goal = fol.parse('Mortal(Socrates)')
clauses = []
for f in prem + [fol.Not(goal)]:
    clauses += fol.to_cnf_clauses(fol.ground(f, ['Socrates']))

M = A.ProofSearchMDP(clauses, entailed=True, step_cost=0.05)
print('clause universe (base + resolution closure):')
for i, c in enumerate(M.universe):
    print(f'  {i}: ' + ('EMPTY' if not c else '{' + ', '.join(sorted(c)) + '}'))
print(f'\n|S| = {len(M.states())}')

clause universe (base + resolution closure):
  0: {Mortal(Socrates), ~Human(Socrates)}
  1: {Human(Socrates)}
  2: {~Mortal(Socrates)}
  3: {Mortal(Socrates)}
  4: {~Human(Socrates)}
  5: EMPTY

|S| = 16


In [14]:
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'V*(s0) = {V[s0]:.3f}\n')
ep = mdp.run_episode(M, mdp.greedy_policy(pi))
for t in ep.transitions:
    print(f'  {str(t.state):16s} {M.describe_action(t.action):34s} r={t.reward:+.2f}')
print(f'\nreturn = {ep.discounted_return():.3f}  '
      f'(1.0 for the verdict, minus {len(ep)-1} resolution steps)')

V*(s0) = 0.900

  {0,1,2}          derive {Mortal(Socrates)}          r=-0.05
  {0,1,2,3}        derive EMPTY                       r=-0.05
  {0,1,2,3,5}      claim:entailed                     r=+1.00

return = 0.900  (1.0 for the verdict, minus 2 resolution steps)


In [15]:
print('random policy value:', round(mdp.policy_value(M, mdp.random_policy(), 400), 3))
print('optimal value      :', round(V[s0], 3))
print('\nA random searcher does far worse than nothing: it claims verdicts it\n'
      'has not justified and is penalised. Requiring the empty clause before\n'
      'crediting a claim is what makes the reward reward *proving*.')

random policy value:

 -0.872
optimal value      : 0.9

A random searcher does far worse than nothing: it claims verdicts it
has not justified and is penalised. Requiring the empty clause before
crediting a claim is what makes the reward reward *proving*.


### Exercise 4.1 — Find the budget at which GEPA stops learning

Sweep `max_metric_calls` and record the held-out score and the rules discovered. Report the smallest budget that finds all five rules.

In [16]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [17]:
rows = []
for budget in [20, 40, 80]:
    t = opt.run_gepa(A.FormalisationProgram(), train, gepa_metric, valset=train,
                     max_metric_calls=budget, reflection_lm=reflect)
    r = opt.compare(A.FormalisationProgram(), t, dev, A.translation_scorer)
    rules = A.FOL_RULEBOOK.active_in(r.instruction_after)
    rows.append({'budget': budget, 'dev_score': r.after['mean_score'],
                 'n_rules': len(rules)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nOptimisation is search under a budget. A run that plateaus may simply\n'
      'have run out of rollouts -- report the budget alongside the score, or the\n'
      'number is uninterpretable.')

2026/08/15 19:51:18 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 20 metric calls of the program. This amounts to 1.67 full evals on the train+val set.


2026/08/15 19:51:18 INFO dspy.teleprompt.gepa.gepa: Using 6 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/20 [00:00<?, ?rollouts/s]

2026/08/15 19:51:18 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 6 (0.0%)


2026/08/15 19:51:18 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0


...

2026/08/15 19:51:19 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/15 19:51:19 INFO dspy.teleprompt.gepa.gepa: Iteration 21: All subsample scores perfect. Skipping.


2026/08/15 19:51:19 INFO dspy.teleprompt.gepa.gepa: Iteration 21: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 78/80 [00:00<00:00, 165.91rollouts/s]

 budget  dev_score  n_rules
     20     0.6250        3
     40     0.8125        4
     80     1.0000        5

Optimisation is search under a budget. A run that plateaus may simply
have run out of rollouts -- report the budget alongside the score, or the
number is uninterpretable.


### Exercise 4.2 — Break the semantic grader

Find a predicted formula that is **not** equivalent to the gold formula in general, but that `equivalent(..., max_size=2)` accepts. Then show a larger `max_size` catching it.

> **Hint.** Reuse Exercise 2.2: how large must a domain be before quantifier order can matter at all?

In [18]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [19]:
# The quantifier-swap pair from Exercise 2.2: on a one-element domain the
# choice of y cannot depend on x, so the two readings coincide.
gold = fol.parse('forall x exists y T(x, y)')
guess = fol.parse('exists y forall x T(x, y)')
print('grader accepts at max_size=1?', A.equivalent(guess, gold, max_size=1))
print('grader accepts at max_size=2?', A.equivalent(guess, gold, max_size=2))
assert A.equivalent(guess, gold, max_size=1)
assert not A.equivalent(guess, gold, max_size=2)
print('\nA grader run at max_size=1 would award full marks to the single worst\n'
      'quantifier error in the chapter. Raising max_size costs exponentially, so\n'
      'the honest report is "equivalent up to size k" -- never "equivalent".\n'
      'Every automated grader has a boundary; the professional move is to state\n'
      'yours rather than to pretend it has none.')

grader accepts at max_size=1? True
grader accepts at max_size=2? False

A grader run at max_size=1 would award full marks to the single worst
quantifier error in the chapter. Raising max_size costs exponentially, so
the honest report is "equivalent up to size k" -- never "equivalent".
Every automated grader has a boundary; the professional move is to state
yours rather than to pretend it has none.


### Exercise 4.3 — Punish unproved verdicts harder

In the proof-search MDP, raise `wrong_verdict_penalty` and find the point at which the optimal policy prefers *not* to answer at all. Discuss what that means for an agent asked a question it cannot settle.

In [20]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [21]:
rows = []
for penalty in [0.0, 1.0, 5.0]:
    Mp = A.ProofSearchMDP(clauses, entailed=False, step_cost=0.05,
                          wrong_verdict_penalty=penalty)
    Vp, pip = mdp.value_iteration(Mp)
    epp = mdp.run_episode(Mp, mdp.greedy_policy(pip))
    rows.append({'penalty': penalty, 'V*': round(Vp[Mp.initial_state()], 3),
                 'steps': len(epp) - 1,
                 'verdict': epp.actions[-1]})
print(pd.DataFrame(rows).to_string(index=False))
print('\nNote we set entailed=False here, so "claim:not-entailed" is correct and\n'
      'needs no proof -- the agent can be right for free. That asymmetry is real:\n'
      'refutation requires a countermodel, but our reward only demands the empty\n'
      'clause for the positive claim. A reward that credits an unjustified\n'
      'negative answer teaches the agent to say "no" when unsure.')

 penalty  V*  steps            verdict
     0.0 1.0      0 claim:not-entailed
     1.0 1.0      0 claim:not-entailed
     5.0 1.0      0 claim:not-entailed

Note we set entailed=False here, so "claim:not-entailed" is correct and
needs no proof -- the agent can be right for free. That asymmetry is real:
refutation requires a countermodel, but our reward only demands the empty
clause for the positive claim. A reward that credits an unjustified
negative answer teaches the agent to say "no" when unsure.


## Chapter 2 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 4 |
|---|---|---|---|
| task | assess an artefact | formalise English | build an axiom |
| MDP | gather evidence | **search for a proof** | construct under constraints |
| grader | labels + judge | **a decision procedure** | labels + profile table |
| GEPA learns | reporting discipline | quantifier semantics | OWL 2 profile limits |

Chapter 2's distinctive contribution is the grader: when your task has a decision procedure, use it — it is cheaper, sharper and more honest than any judge. Chapter 3 asks what happens when you *design a language* so that such a procedure always exists.